# anon_check
Python translation of `anon_check.Rmd`: k-anonymity, local suppression and l-diversity.

Requires only `pandas` (`pip install pandas`).

In [1]:
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## Load data

In [5]:
df = pd.read_csv(r"..\data\private_data_preprocessed.csv", dtype=str)   # read everything as text (evote, zip as characters)
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed") or c == "X"], errors="ignore")  # drop row-index column
df["zip"] = df["zip"].str[0] + "***"
df

,name,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,*,Female,0,2***,Medium,Danish,Never married,Green,18-29
1,*,Female,0,2***,Medium,Danish,Never married,Red,45-64
2,*,Female,1,2***,High,Danish,Never married,Green,45-64
3,*,Male,1,2***,High,Danish,Never married,Red,65 or older
4,*,Male,0,2***,Medium,Danish,Never married,Green,30-44
5,*,Female,0,2***,High,Danish,Never married,Green,30-44
6,*,Male,0,2***,High,Danish,Never married,Green,18-29
7,*,Male,0,2***,Medium,Danish,Married/separated,Red,45-64
8,*,Female,0,2***,High,Danish,Never married,Red,45-64
9,*,Male,0,2***,Medium,Danish,Divorced,Green,45-64


## Quasi-identifiers and sensitive attributes

In [6]:
qi   = ["sex", "evote", "zip", "education", "citizenship", "age_group"]
sens = ["party", "marital_status"]

## k-anonymity check
Size of each equivalence class. `dropna=False` keeps suppressed (`NaN`) values as their own category, like sdcMicro does.

In [7]:
def k_groups(data, qi):
    return data.groupby(qi, dropna=False).size().reset_index(name="n")

K_TARGET = 3

groups = k_groups(df, qi)
print("current k:", groups["n"].min())
print(f"groups violating k = {K_TARGET}:", (groups["n"] < K_TARGET).sum())
print(f"records violating k = {K_TARGET}:", groups.loc[groups["n"] < K_TARGET, "n"].sum())

current k: 1
groups violating k = 3: 35
records violating k = 3: 50


## Local suppression to reach k
sdcMicro's `kAnon()` has no Python equivalent, so this is a simple greedy version: 
for records in classes smaller than k, set one quasi-identifier to `NaN`, starting with the least important column, and repeat until every class has at least k records.

`importance` lists the columns in the order they get suppressed (first = suppressed first). I put the columns with the most categories first, since they cause the most violations. Change it to whatever you can best afford to lose.

In [8]:
def k_anon_suppress(data, qi, k, importance=None):
    out = data.copy()
    order = importance or sorted(qi, key=lambda c: -out[c].nunique())   # most categories first
    for col in order:
        size = out.groupby(qi, dropna=False)[qi[0]].transform("size")
        viol = size < k
        if not viol.any():
            break
        out.loc[viol, col] = pd.NA
    # records still in a class < k: suppress all their quasi-identifiers so they merge into one class
    size = out.groupby(qi, dropna=False)[qi[0]].transform("size")
    out.loc[size < k, qi] = pd.NA
    size = out.groupby(qi, dropna=False)[qi[0]].transform("size")
    if (size < k).any():
        print(f"warning: {(size < k).sum()} fully suppressed records still form a class < k={k}; drop them or lower k")
    return out

df_anon = k_anon_suppress(df, qi, k=K_TARGET)

print("k after suppression:", k_groups(df_anon, qi)["n"].min())
print("NaNs introduced per column:")
print(df_anon[qi].isna().sum())
df_anon

k after suppression: 2
NaNs introduced per column:
sex             2
evote           2
zip             2
education      19
citizenship     2
age_group      50
dtype: int64


,name,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,*,Female,0,2***,Medium,Danish,Never married,Green,18-29
1,*,Female,0,2***,Medium,Danish,Never married,Red,45-64
2,*,Female,1,2***,High,Danish,Never married,Green,45-64
3,*,Male,1,2***,High,Danish,Never married,Red,65 or older
4,*,Male,0,2***,Medium,Danish,Never married,Green,30-44
5,*,Female,0,2***,High,Danish,Never married,Green,30-44
6,*,Male,0,2***,High,Danish,Never married,Green,18-29
7,*,Male,0,2***,Medium,Danish,Married/separated,Red,45-64
8,*,Female,0,2***,High,Danish,Never married,Red,45-64
9,*,Male,0,2***,Medium,Danish,Divorced,Green,45-64


## l-diversity check
Per equivalence class: size, joint l (distinct `(party, marital_status)` pairs), and l for each sensitive attribute alone.

In [9]:
def l_diversity(data, qi, sens):
    g = data.groupby(qi, dropna=False)
    res = g.agg(n=(sens[0], "size")).reset_index()
    res["l_joint"] = g.apply(lambda x: len(x[sens].drop_duplicates()), include_groups=False).values
    res["l_party"] = g["party"].nunique(dropna=False).values
    res["l_marital"] = g["marital_status"].nunique(dropna=False).values
    return res

def l_report(data, qi, sens, l=2):
    lg = l_diversity(data, qi, sens)
    print("current l (joint):           ", lg["l_joint"].min())
    print("groups violating l (joint):  ", (lg["l_joint"]   < l).sum())
    print("groups violating l (party):  ", (lg["l_party"]   < l).sum())
    print("groups violating l (marital):", (lg["l_marital"] < l).sum())

    records = pd.Series({
        "records_total":        lg["n"].sum(),
        "records_viol_joint":   lg.loc[lg["l_joint"]   < l, "n"].sum(),
        "records_viol_party":   lg.loc[lg["l_party"]   < l, "n"].sum(),
        "records_viol_marital": lg.loc[lg["l_marital"] < l, "n"].sum(),
        "records_viol_any":     lg.loc[(lg["l_party"] < l) | (lg["l_marital"] < l), "n"].sum(),
    })
    print()
    print(records.to_string())
    return lg

L_TARGET = 2

### On the original data (`df`)

In [10]:
l_groups = l_report(df, qi, sens, L_TARGET)

# the offending groups, smallest first
l_groups[l_groups["l_joint"] < L_TARGET].sort_values(["l_joint", "n"])

current l (joint):            1
groups violating l (joint):   31
groups violating l (party):   38
groups violating l (marital): 35

records_total           196
records_viol_joint       44
records_viol_party       79
records_viol_marital     55
records_viol_any         90


,sex,evote,zip,education,citizenship,age_group,n,l_joint,l_party,l_marital
4,Female,0,2***,High,non-danish,18-29,1,1,1,1
5,Female,0,2***,High,non-danish,30-44,1,1,1,1
11,Female,0,2***,Low,non-danish,45-64,1,1,1,1
12,Female,0,2***,Low,non-danish,65 or older,1,1,1,1
17,Female,0,2***,Medium,non-danish,30-44,1,1,1,1
18,Female,0,2***,Medium,non-danish,65 or older,1,1,1,1
19,Female,1,2***,High,Danish,18-29,1,1,1,1
22,Female,1,2***,High,non-danish,30-44,1,1,1,1
23,Female,1,2***,High,non-danish,45-64,1,1,1,1
26,Female,1,2***,Low,non-danish,65 or older,1,1,1,1


### On the k-anonymized data (`df_anon`)

In [11]:
l_groups_anon = l_report(df_anon, qi, sens, L_TARGET)

l_groups_anon[l_groups_anon["l_joint"] < L_TARGET].sort_values(["l_joint", "n"])

current l (joint):            1
groups violating l (joint):   2
groups violating l (party):   12
groups violating l (marital): 4

records_total           196
records_viol_joint        6
records_viol_party       50
records_viol_marital     13
records_viol_any         57


,sex,evote,zip,education,citizenship,age_group,n,l_joint,l_party,l_marital
8,Female,0,2***,Medium,Danish,18-29,3,1,1,1
19,Male,0,2***,High,Danish,18-29,3,1,1,1


### Stricter view (per-attribute)
A group passes only if **every** sensitive attribute is diverse. This is the number I'd report as "groups failing l-diversity".

In [12]:
l_groups_anon["l_min"] = l_groups_anon[["l_party", "l_marital"]].min(axis=1)
failing = l_groups_anon[l_groups_anon["l_min"] < L_TARGET].sort_values(["l_min", "n"])
print("groups failing on at least one attribute:", len(failing))
print("records in those groups:", failing["n"].sum())
failing

groups failing on at least one attribute: 14
records in those groups: 57


,sex,evote,zip,education,citizenship,age_group,n,l_joint,l_party,l_marital,l_min
39,NaN,NaN,NaN,NaN,NaN,NaN,2,2,1,2,1
0,Female,0,2***,High,Danish,18-29,3,2,2,1,1
3,Female,0,2***,High,Danish,65 or older,3,2,1,2,1
6,Female,0,2***,Low,Danish,45-64,3,2,1,2,1
8,Female,0,2***,Medium,Danish,18-29,3,1,1,1,1
19,Male,0,2***,High,Danish,18-29,3,1,1,1,1
24,Male,0,2***,Low,Danish,18-29,3,2,1,2,1
26,Male,0,2***,Low,Danish,NaN,3,2,1,2,1
15,Female,1,2***,Low,Danish,NaN,4,2,1,2,1
27,Male,0,2***,Medium,Danish,18-29,4,2,2,1,1


### Skew check
Max share of the most common value in each group. A share close to 1 means an attacker can guess that attribute with high confidence even if l is technically 2.

In [13]:
def max_share(s):
    return s.value_counts(dropna=False).iloc[0] / len(s)

skew = (df_anon.groupby(qi, dropna=False)
        .agg(n=("party", "size"),
             max_share_party=("party", max_share),
             max_share_marital=("marital_status", max_share))
        .reset_index())

skew[(skew["max_share_party"] > 0.8) | (skew["max_share_marital"] > 0.8)].sort_values("n")

,sex,evote,zip,education,citizenship,age_group,n,max_share_party,max_share_marital
39,NaN,NaN,NaN,NaN,NaN,NaN,2,1.000000,0.500000
0,Female,0,2***,High,Danish,18-29,3,0.666667,1.000000
3,Female,0,2***,High,Danish,65 or older,3,1.000000,0.666667
6,Female,0,2***,Low,Danish,45-64,3,1.000000,0.666667
8,Female,0,2***,Medium,Danish,18-29,3,1.000000,1.000000
19,Male,0,2***,High,Danish,18-29,3,1.000000,1.000000
24,Male,0,2***,Low,Danish,18-29,3,1.000000,0.666667
26,Male,0,2***,Low,Danish,NaN,3,1.000000,0.666667
15,Female,1,2***,Low,Danish,NaN,4,1.000000,0.500000
27,Male,0,2***,Medium,Danish,18-29,4,0.750000,1.000000
